# GNNHAR-IV Full Colab Run

This notebook runs the full GNNHAR-IV empirical analysis on Colab. Code and input data are cloned from the private GitHub repository, so no Google Drive mount or Drive API service account is required. Outputs are written locally under `/content/gnnhar_outputs` and then committed back to the private GitHub repository under `outputs/colab-runs/<run_id>/`.

Before running, add one Colab Secret:

- `GITHUB_TOKEN`: a GitHub token with read/write access to the private `easygl1der/GNNHAR` repository.

In [ ]:
import json
import pathlib
import shutil
import subprocess
import sys
from datetime import datetime, timezone

from google.colab import userdata

REPO_OWNER = 'easygl1der'
REPO_NAME = 'GNNHAR'
BRANCH = '2026-06-01'
REPO_DIR = pathlib.Path('/content/GNNHAR')
OUTPUT_DIR = pathlib.Path('/content/gnnhar_outputs')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
REPO_OUTPUT_DIR = REPO_DIR / 'outputs' / 'colab-runs' / RUN_ID

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    raise RuntimeError('Missing Colab Secret: GITHUB_TOKEN')

netrc_path = pathlib.Path.home() / '.netrc'
netrc_path.write_text(f'machine github.com login x-access-token password {GITHUB_TOKEN}\n')
netrc_path.chmod(0o600)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

repo_url = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'
subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', BRANCH, repo_url, str(REPO_DIR),
], check=True)

print('cloned:', REPO_DIR)
print('output_dir:', OUTPUT_DIR)
print('repo_output_dir:', REPO_OUTPUT_DIR)

In [ ]:
import subprocess
import sys

mods = ['numpy', 'pandas', 'sklearn', 'matplotlib', 'torch', 'scipy']
packages = {'sklearn': 'scikit-learn'}
missing = []
for mod in mods:
    try:
        __import__(mod)
    except Exception:
        missing.append(mod)
print('missing:', missing)
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *[packages.get(m, m) for m in missing]])

In [ ]:
subprocess.run([
    'python', str(REPO_DIR / 'scripts/analysis/gnnhar_iv_pipeline.py'),
    '--data-dir', str(REPO_DIR / 'experiments/dow30/data'),
    '--output-dir', str(OUTPUT_DIR),
    '--epochs', '250',
], check=True)

In [ ]:
import json

for path in sorted(OUTPUT_DIR.glob('**/*')):
    if path.is_file():
        print(path)

print('\nmetadata:')
print(json.dumps(json.loads((OUTPUT_DIR / 'run_metadata.json').read_text()), indent=2)[:2000])

In [ ]:
import pandas as pd

pd.read_csv(OUTPUT_DIR / 'tables/model_losses.csv').head(20)

In [ ]:
import shutil
import subprocess

if REPO_OUTPUT_DIR.exists():
    shutil.rmtree(REPO_OUTPUT_DIR)
REPO_OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(OUTPUT_DIR, REPO_OUTPUT_DIR)

relative_output = REPO_OUTPUT_DIR.relative_to(REPO_DIR)
subprocess.run(['git', '-C', str(REPO_DIR), 'config', 'user.email', 'colab-runner@users.noreply.github.com'], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'config', 'user.name', 'Colab Runner'], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'add', str(relative_output)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'commit', '-m', f'Add Colab GNNHAR-IV outputs {RUN_ID}'], check=True)
try:
    subprocess.run(['git', '-C', str(REPO_DIR), 'push', 'origin', f'HEAD:{BRANCH}'], check=True)
except subprocess.CalledProcessError:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--rebase', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'push', 'origin', f'HEAD:{BRANCH}'], check=True)

print('Pushed GitHub output path:', relative_output)
print('GitHub URL:', f'https://github.com/{REPO_OWNER}/{REPO_NAME}/tree/{BRANCH}/{relative_output}')